In [1]:
import math
from collections import Counter, defaultdict

train_file = "en_ewt-ud-train.conllu"
test_file = "en_ewt-ud-test.conllu"

def load_data(file):
    data = []
    words = []
    tags = []

    for line in open(file, encoding="utf-8"):
        line = line.strip()

        if not line:
            if words:
                data.append((words, tags))
                words, tags = [], []
            continue

        if line.startswith("#"):
            continue

        x = line.split("\t")

        if len(x) == 10 and "-" not in x[0] and "." not in x[0]:
            words.append(x[1])
            tags.append(x[3])

    return data


train = load_data(train_file)
test = load_data(test_file)

# Transition and emission counts
trans = defaultdict(Counter)
emit = defaultdict(Counter)

for words, tags in train:
    prev = "START"

    for word, tag in zip(words, tags):
        trans[prev][tag] += 1
        emit[tag][word.lower()] += 1
        prev = tag

    trans[prev]["END"] += 1

# Convert counts to probabilities
for p in trans:
    total = sum(trans[p].values())
    for t in trans[p]:
        trans[p][t] /= total

for t in emit:
    total = sum(emit[t].values())
    for w in emit[t]:
        emit[t][w] /= total

tags = list(emit)


# Viterbi
def viterbi(words):
    V = [{}]
    B = [{}]

    for t in tags:
        V[0][t] = math.log(trans["START"].get(t, 1e-6)) + \
                  math.log(emit[t].get(words[0].lower(), 1e-6))
        B[0][t] = None

    for i in range(1, len(words)):
        V.append({})
        B.append({})

        for t in tags:
            best = max(
                (V[i-1][p] +
                 math.log(trans[p].get(t, 1e-6)) +
                 math.log(emit[t].get(words[i].lower(), 1e-6)), p)
                for p in tags
            )

            V[i][t], B[i][t] = best

    last = max(
        tags,
        key=lambda t: V[-1][t] +
        math.log(trans[t].get("END", 1e-6))
    )

    result = [last]

    for i in range(len(words)-1, 0, -1):
        result.append(B[i][result[-1]])

    return result[::-1]


# User input
sentence = input("Enter a sentence: ")
words = sentence.split()
predicted = viterbi(words)

print("\nPOS Tags:")
for word, tag in zip(words, predicted):
    print(word, "→", tag)


# Test accuracy
correct = 0
total = 0

for words, actual in test:
    predicted = viterbi(words)

    for p, a in zip(predicted, actual):
        correct += p == a
        total += 1

print("\nTest Accuracy:", round(correct / total * 100, 2), "%")

Enter a sentence:  NLP is natural language processig



POS Tags:
NLP → PRON
is → AUX
natural → ADJ
language → NOUN
processig → PUNCT

Test Accuracy: 88.8 %
